In [1]:
import numpy as np
import gdspy
import os
from autograd import numpy as npa

################################################### design parameter #########################
resolution = 50  # 해상도
Mask_thick = 25
design_region_x = round(0.4, 2)
design_region_y = round(7.0 + 2 * Mask_thick / resolution, 2)
design_region_z = round(7.0 + 2 * Mask_thick / resolution, 2)
design_region_resolution = int(resolution)
Nx = int(design_region_resolution * design_region_x) + 1
Ny = int(design_region_resolution * design_region_y) + 1
Nz = int(design_region_resolution * design_region_z) + 1
##############################################################################################

# Numpy 배열 생성 2D 배열을 만듭니다
structure_weight = np.loadtxt('lastdesign.txt')
structure_weight = structure_weight.reshape(Nx, Ny * Nz)[Nx - 1]
data = npa.rot90(structure_weight.reshape(Ny, Nz))  # ZxY

# Check if the cell 'TOP_RECT' and 'TOP_SMOOTH' exist and delete them if they do
if 'TOP_RECT' in gdspy.current_library.cells:
    del gdspy.current_library.cells['TOP_RECT']
if 'TOP_SMOOTH' in gdspy.current_library.cells:
    del gdspy.current_library.cells['TOP_SMOOTH']

# Create the new cells
rect_cell = gdspy.Cell('TOP_RECT')
smooth_cell = gdspy.Cell('TOP_SMOOTH')

# Rectangle dimensions
width = 1
height = 1

# Numpy 배열을 기반으로 GDS의 경로(Path) 생성
rectangles = []
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        if data[i, j] == 1:
            # 각 사각형의 좌표를 정밀하게 조정
            x0 = j * width
            y0 = -i * height
            x1 = (j + 1) * width
            y1 = -(i + 1) * height
            rectangle = gdspy.Rectangle((x0, y0), (x1, y1))
            rectangles.append(rectangle)

def moving_average(points, window_size=1):
    # 좌표의 이동 평균을 계산
    smoothed_points = []
    for i in range(len(points)):
        start_idx = max(0, i - window_size // 2)
        end_idx = min(len(points), i + window_size // 2 + 1)
        avg_x = np.mean([p[0] for p in points[start_idx:end_idx]])
        avg_y = np.mean([p[1] for p in points[start_idx:end_idx]])
        smoothed_points.append((avg_x, avg_y))
    return smoothed_points

def reduce_points(points, max_points):
    # 점의 수를 최대 허용 점 개수에 맞게 줄이는 함수
    if len(points) <= max_points:
        return points
    step = len(points) // max_points
    return points[::step]  # step 간격으로 점을 선택하여 줄임

# Rectangle 방식으로 셀에 추가
if rectangles:
    for rectangle in rectangles:
        rect_cell.add(rectangle)

# Smooth 방식으로 다각형 병합 및 처리
if rectangles:
    # 병합된 다각형을 생성
    # 직사각형을 살짝 확장하여 작은 간격을 메움
    expanded_rectangles = [gdspy.offset(rectangle, distance=0.05) for rectangle in rectangles]

    # 확장된 직사각형들을 병합
    merged_polygon = gdspy.boolean(expanded_rectangles, None, 'or', precision=1e-5, max_points=1000000)

    # 병합된 다각형이 존재할 때 처리
    if merged_polygon:
        polygons_to_save = []
        for polygon_boundary in merged_polygon.polygons:
            # 이동 평균을 사용해 좌표를 매끄럽게 처리
            smoothed_points = moving_average(polygon_boundary, window_size=2)
            reduced_points = reduce_points(smoothed_points, max_points=10000)  # 필요에 따라 max_points 조정
            smoothed_polygon = gdspy.Polygon(reduced_points)
            polygons_to_save.append(smoothed_polygon)
        
        # 다각형을 여러 개의 파일로 저장 (예시로 1000개씩 분할 저장)
        for idx, polygon in enumerate(polygons_to_save):
            if idx % 1000 == 0:
                # 새로운 파일에 추가
                gdspy.write_gds(f'output_smooth_{idx // 1000}.gds', cells=[smooth_cell])
            smooth_cell.add(polygon)

# Save the GDS files for both cases
gdspy.write_gds('output_rect.gds', cells=[rect_cell])
gdspy.write_gds('output_smooth.gds', cells=[smooth_cell])

# Optionally, save images of the cells as SVG.
rect_cell.write_svg('output_rect.svg')
smooth_cell.write_svg('output_smooth.svg')

# Display all cells using the internal viewer.
gdspy.LayoutViewer()


/tmp/ipykernel_1014727/1909470013.py:108: DeprecationWarning: [GDSPY] Use of the global library is deprecated.  Pass LayoutViewer a GdsLibrary instance.
  gdspy.LayoutViewer()


<gdspy.viewer.LayoutViewer object .!layoutviewer>